In [ ]:
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm
from sklearn.model_selection import train_test_split


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

ROIS_ROOT   = r"C:\vs code\PCB_circuit\PCB_DATASET\PCB_DATASET\roi_Output1"  # ROI folder
LABELS_CSV  = r"C:\vs code\PCB_circuit\PCB_DATASET\PCB_DATASET\roi_labels1.csv"
OUT_DIR     = r"C:\vs code\PCB_circuit\PCB_circuit_output"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

IMG_SIZE      = 224
BATCH_SIZE    = 4
EPOCHS        = 5
LR            = 0.001
WEIGHT_DECAY  = 1e-4
PATIENCE      = 4
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS   = 0  # safe for Windows
PIN_MEMORY    = torch.cuda.is_available()
MODEL_PATH    = os.path.join(OUT_DIR, "pcb_model.pth")


c:\Users\manne\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(f"Using device: {DEVICE}")

Using device: cpu


In [ ]:

class ROIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, class_to_idx=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        # Build class mapping if not provided
        if class_to_idx is None:
            classes = sorted(self.df["label"].unique().tolist())
            self.class_to_idx = {c: i for i, c in enumerate(classes)}
        else:
            self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["abs_path"]
        label = row["label"]
        y = self.class_to_idx[label]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, y


def build_transforms(img_size=IMG_SIZE):
    mean = (0.485, 0.456, 0.406)
    std  = (0.229, 0.224, 0.225)

    train_tfms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    val_tfms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    return train_tfms, val_tfms


In [ ]:

df = pd.read_csv(LABELS_CSV)
df = df.rename(columns={"file_name": "filename"})  # ensure column name consistency

# Build absolute paths including label subfolder
df["abs_path"] = df.apply(lambda row: os.path.join(ROIS_ROOT, row["label"], row["filename"]), axis=1)

# Check missing files
missing = df[~df["abs_path"].apply(os.path.exists)]
if not missing.empty:
    print("⚠ Missing files detected:\n", missing)
else:
    print("✅ All files exist.")


train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
val_df, test_df  = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")


✅ All files exist.
Train: 2200, Val: 471, Test: 472


In [ ]:
train_tfms, val_tfms = build_transforms(IMG_SIZE)

train_ds = ROIDataset(train_df, transform=train_tfms)
val_ds   = ROIDataset(val_df, transform=val_tfms, class_to_idx=train_ds.class_to_idx)
test_ds  = ROIDataset(test_df, transform=val_tfms, class_to_idx=train_ds.class_to_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

class_to_idx = train_ds.class_to_idx
with open(os.path.join(OUT_DIR, "class_to_idx.json"), "w") as f:
    import json
    json.dump(class_to_idx, f)
idx_to_class = {v:k for k,v in class_to_idx.items()}


In [ ]:

def build_model(num_classes):
    print(f"[INFO] Building EfficientNet-B4 with {num_classes} classes...")
    model = timm.create_model("efficientnet_b4", pretrained=True, num_classes=num_classes)
    return model

model = build_model(num_classes=len(class_to_idx))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


[INFO] Building EfficientNet-B4 with 6 classes...


In [9]:
best_val_acc = 0.0
patience_ct = 0

for epoch in range(1, EPOCHS+1):
    # ---------- Train ----------
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # ---------- Validation ----------
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # ---------- Early Stopping ----------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        patience_ct = 0
        print("✅ New best model saved!")
    else:
        patience_ct += 1
        if patience_ct >= PATIENCE:
            print("[STOP] Early stopping triggered.")
            break


Epoch [1/5] | Train Loss: 0.8669 | Train Acc: 0.6377 | Val Acc: 0.7558
✅ New best model saved!
Epoch [2/5] | Train Loss: 0.6169 | Train Acc: 0.7464 | Val Acc: 0.7771
✅ New best model saved!
Epoch [3/5] | Train Loss: 0.5427 | Train Acc: 0.7645 | Val Acc: 0.8174
✅ New best model saved!
Epoch [4/5] | Train Loss: 0.4646 | Train Acc: 0.7977 | Val Acc: 0.8153
Epoch [5/5] | Train Loss: 0.4374 | Train Acc: 0.8164 | Val Acc: 0.7856


In [ ]:

import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm
from sklearn.model_selection import train_test_split
import json


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


ROIS_ROOT   = r"C:\vs code\PCB_circuit\PCB_DATASET\PCB_DATASET\roi_Output1"
LABELS_CSV  = r"C:\vs code\PCB_circuit\PCB_DATASET\PCB_DATASET\roi_labels1.csv"
OUT_DIR     = r"C:\vs code\PCB_circuit\PCB_circuit_output"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

MODEL_PATH  = os.path.join(OUT_DIR, "pcb_model.pth")


IMG_SIZE      = 224
BATCH_SIZE    = 4
EPOCHS        = 5
LR            = 0.001
WEIGHT_DECAY  = 1e-4
PATIENCE      = 4
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS   = 0
PIN_MEMORY    = torch.cuda.is_available()


In [ ]:


class ROIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, class_to_idx=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        if class_to_idx is None:
            classes = sorted(self.df["label"].unique().tolist())
            self.class_to_idx = {c: i for i, c in enumerate(classes)}
        else:
            self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["abs_path"]
        label = row["label"]
        y = self.class_to_idx[label]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, y


def build_transforms(img_size=IMG_SIZE):
    mean = (0.485, 0.456, 0.406)
    std  = (0.229, 0.224, 0.225)

    train_tfms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    val_tfms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    return train_tfms, val_tfms


In [ ]:


df = pd.read_csv(LABELS_CSV)
df = df.rename(columns={"file_name": "filename"})
df["abs_path"] = df.apply(lambda row: os.path.join(ROIS_ROOT, row["label"], row["filename"]), axis=1)

missing = df[~df["abs_path"].apply(os.path.exists)]
if not missing.empty:
    print("⚠ Missing files detected:\n", missing)
else:
    print("✅ All files exist.")


train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
val_df, test_df  = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")


✅ All files exist.
Train: 2200, Val: 471, Test: 472


In [ ]:

train_df.to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(OUT_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)

train_tfms, val_tfms = build_transforms(IMG_SIZE)

train_ds = ROIDataset(train_df, transform=train_tfms)
val_ds   = ROIDataset(val_df, transform=val_tfms, class_to_idx=train_ds.class_to_idx)
test_ds  = ROIDataset(test_df, transform=val_tfms, class_to_idx=train_ds.class_to_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# Save class mapping
class_to_idx = train_ds.class_to_idx
with open(os.path.join(OUT_DIR, "class_to_idx.json"), "w") as f:
    json.dump(class_to_idx, f)
idx_to_class = {v:k for k,v in class_to_idx.items()}


In [ ]:


def build_model(num_classes):
    print(f"[INFO] Building EfficientNet-B4 with {num_classes} classes...")
    model = timm.create_model("efficientnet_b4", pretrained=True, num_classes=num_classes)
    return model

model = build_model(num_classes=len(class_to_idx))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

start_epoch = 0
best_val_acc = 0.0

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print(f"✅ Loaded previous model from {MODEL_PATH}")


[INFO] Building EfficientNet-B4 with 6 classes...


C:\Users\manne\AppData\Local\Temp\ipykernel_10708\502570988.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=DE

✅ Loaded previous model from C:\vs code\PCB_circuit\PCB_circuit_output\pcb_model.pth


In [ ]:

patience_ct = 0

for epoch in range(start_epoch+1, EPOCHS+1):
    # ---------- Train ----------
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # ---------- Validation ----------
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # ---------- Early Stopping ----------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        patience_ct = 0
        print("✅ New best model saved!")
    else:
        patience_ct += 1
        if patience_ct >= PATIENCE:
            print("[STOP] Early stopping triggered.")
            break


Epoch [1/5] | Train Loss: 0.0952 | Train Acc: 0.9627 | Val Acc: 0.8514
Epoch [2/5] | Train Loss: 0.0899 | Train Acc: 0.9677 | Val Acc: 0.8471
Epoch [3/5] | Train Loss: 0.0917 | Train Acc: 0.9650 | Val Acc: 0.8344
Epoch [4/5] | Train Loss: 0.0699 | Train Acc: 0.9741 | Val Acc: 0.8535
[STOP] Early stopping triggered.
